# Customer Churn Prediction — Model Training

This notebook trains and compares Logistic Regression and Random Forest models.
The target is `Churn`, where Yes = 1 and No = 0.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).drop_duplicates().reset_index(drop=True)
df = df.drop(columns=['customerID'])
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Training rows:', len(X_train))
print('Test rows:', len(X_test))

In [ ]:
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = X.select_dtypes(exclude=['number']).columns.tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical)
])

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)
}

results = {}
for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    results[name] = {'model': pipeline, 'roc_auc': roc_auc_score(y_test, probabilities)}
    print(f'\n{name}')
    print(classification_report(y_test, predictions, target_names=['No Churn', 'Churn']))
    print('Confusion matrix:')
    print(confusion_matrix(y_test, predictions))
    print(f'ROC-AUC: {results[name]["roc_auc"]:.4f}')

## Interpretation

After running the notebook, compare the models using recall, F1-score, and ROC-AUC.
For churn prediction, recall is important because failing to identify a customer who is likely to churn can represent a missed retention opportunity.